Comparações espaciais: abordagem, período e fonte

In [ ]:
from pathlib import Path
import re
import sys
import pandas as pd

sys.path.append("/code/scripts")

from analysis_config import ANALYSIS, structural_pairs
from comparison_utils import compare_classes

In [ ]:
pilots = ["fundao", "badajoz"]
all_summary = []
matrices = {}

for pilot in pilots:
    pilot_dir = ANALYSIS / "02_classes" / pilot
    inventory = pd.read_csv(
        pilot_dir / f"class_inventory_{pilot}.csv"
    )

    paths = dict(zip(
        inventory["map_id"],
        inventory["local_class_raster"]
    ))

    out_dir = ANALYSIS / "04_spatial_comparisons" / pilot
    raster_dir = out_dir / "difference_rasters"
    raster_dir.mkdir(parents=True, exist_ok=True)

    for pair in structural_pairs(pilot):
        output = raster_dir / f"{pair['comparison_id']}.tif"

        summary, matrix = compare_classes(
            paths[pair["map_a"]],
            paths[pair["map_b"]],
            output,
            pilot=pilot,
            **pair
        )

        all_summary.append(summary)
        matrices[f"{pilot}_{pair['comparison_id']}"] = matrix

In [ ]:
summary = pd.concat(all_summary, ignore_index=True)
out_dir = ANALYSIS / "04_spatial_comparisons"
out_dir.mkdir(parents=True, exist_ok=True)
output = out_dir / "spatial_comparisons.xlsx"

with pd.ExcelWriter(output) as writer:
    summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    used_names = set()

    for name, matrix in matrices.items():
        clean = re.sub(r"[^A-Za-z0-9_]", "_", name)[:31]
        sheet = clean
        suffix = 1

        while sheet in used_names:
            suffix += 1
            sheet = f"{clean[:27]}_{suffix}"

        used_names.add(sheet)
        matrix.to_excel(writer, sheet_name=sheet)

print("Resultados guardados em:", output)
summary